In [ ]:
import os
import yt_dlp
from pydub import AudioSegment

# === CONFIG ===
YOUTUBE_URL = "https://www.youtube.com/example"
OUTPUT_DIR = "output"
SEGMENTS_DIR = os.path.join(OUTPUT_DIR, "segments")

START_OFFSET = 9 * 1000       # 9 saniye
SEGMENT_LENGTH = 11 * 1000    # 11 saniye
STEP = 13 * 1000              # 13 saniye

os.makedirs(SEGMENTS_DIR, exist_ok=True)

# === 1. VIDEO DOWNLOAD ===
def download_audio():
    
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{OUTPUT_DIR}/audio.%(ext)s',
        'cookiefile': 'cookies.txt',

        # 🔥 KRİTİK FIXLER
        'js_runtimes': {
            'node': {}
        },
        'remote_components': ['ejs:github'],

        # opsiyonel ama iyi
        'geo_bypass': True,

        # ffmpeg path (eğer PATH'e eklemezsen)
        'ffmpeg_location': 'C:\\ffmpeg\\bin',

        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
        }],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YOUTUBE_URL])

    return f"{OUTPUT_DIR}/audio.wav"


# === 2. SEGMENTATION ===
def create_segments(audio_path):
    audio = AudioSegment.from_wav(audio_path)
    duration = len(audio)

    segments_info = []

    current = START_OFFSET
    index = 1

    while current + SEGMENT_LENGTH < duration:
        segment = audio[current:current + SEGMENT_LENGTH]

        filename = f"{SEGMENTS_DIR}/segment_{index}.wav"
        segment.export(filename, format="wav")

        start_sec = current / 1000
        end_sec = (current + SEGMENT_LENGTH) / 1000

        segments_info.append({
            "index": index,
            "file": filename,
            "start": start_sec,
            "end": end_sec
        })

        current += STEP
        index += 1

    return segments_info


# === 3. SAVE TIMESTAMPS ===
def save_timestamps(segments_info):
    txt_path = os.path.join(OUTPUT_DIR, "timestamps.txt")

    with open(txt_path, "w", encoding="utf-8") as f:
        for seg in segments_info:
            f.write(f"{seg['index']}. {seg['start']:.2f}s - {seg['end']:.2f}s\n")

    print(f"Timestamps saved: {txt_path}")


# === RUN ===
if __name__ == "__main__":
    audio_path = download_audio()
    segments = create_segments(audio_path)
    save_timestamps(segments)

    print(f"Total segments: {len(segments)}")

[youtube] Extracting URL: https://www.youtube.com/watch?v=1wKRcLsGfBg
[youtube] 1wKRcLsGfBg: Downloading webpage
[youtube] 1wKRcLsGfBg: Downloading android vr player API JSON
[youtube] 1wKRcLsGfBg: Downloading web embedded client config
[youtube] 1wKRcLsGfBg: Downloading player 8e54e4ea-main
[youtube] 1wKRcLsGfBg: Downloading web embedded player API JSON
[youtube] [jsc:node] Solving JS challenges using node
[youtube] 1wKRcLsGfBg: Downloading m3u8 information
[info] 1wKRcLsGfBg: Downloading 1 format(s): 251
[download] output\audio.webm has already been downloaded
[download] 100% of   18.67MiB
[ExtractAudio] Destination: output\audio.wav
Deleting original file output\audio.webm (pass -k to keep)
Timestamps saved: output\timestamps.txt
Total segments: 100


In [21]:
import requests
import json
import base64
import hashlib
import hmac
import time
from urllib.parse import urlencode

# === CONFIG ===
HOST = "identify-ap-southeast-1.acrcloud.com"  # ex: identify-eu-west-1.acrcloud.com
ACCESS_KEY = "c190d1cfdc7fdd482567d1e56a7b33fe"
ACCESS_SECRET = "GfpVh5An21PiM3deCTNV50ayJCdB5tNSzSURH1KJ"

def recognize_file(file_path):
    http_method = "POST"
    http_uri = "/v1/identify"
    data_type = "audio"
    signature_version = "1"
    timestamp = str(int(time.time()))

    string_to_sign = "\n".join([http_method, http_uri, ACCESS_KEY, data_type, signature_version, timestamp])
    sign = base64.b64encode(hmac.new(ACCESS_SECRET.encode('utf-8'), string_to_sign.encode('utf-8'), digestmod=hashlib.sha1).digest()).decode('utf-8')

    files = {'sample': open(file_path, 'rb')}
    payload = {
        'access_key': ACCESS_KEY,
        'data_type': data_type,
        'signature_version': signature_version,
        'signature': sign,
        'timestamp': timestamp
    }

    response = requests.post(f'https://{HOST}{http_uri}', files=files, data=payload)
    data = response.json()

    if 'metadata' in data and 'music' in data['metadata']:
        music = data['metadata']['music'][0]
        song_name = music.get('title', 'UNKNOWN')
        artist = music.get('artists', [{'name':'UNKNOWN'}])[0]['name']
        return song_name, artist, True
    else:
        return 'UNKNOWN', 'UNKNOWN', False

# === TEST ===
song, artist, ok = recognize_file("output/segments/segment_1.wav")
print(song, artist, ok)

XL (Album Version) Nil Karaibrahimgil True


In [22]:
import os

# === CONFIG ===
SEGMENTS_DIR = "output/segments"
OUTPUT_TXT = "output/songs_detected.txt"

HOST = "identify-ap-southeast-1.acrcloud.com"  # ex: identify-eu-west-1.acrcloud.com
ACCESS_KEY = "c190d1cfdc7fdd482567d1e56a7b33fe"
ACCESS_SECRET = "GfpVh5An21PiM3deCTNV50ayJCdB5tNSzSURH1KJ"

# Tanıma fonksiyonu (önceden yazdığın)
def recognize_file(file_path):
    import requests, json, base64, hashlib, hmac, time
    from urllib.parse import urlencode

    http_method = "POST"
    http_uri = "/v1/identify"
    data_type = "audio"
    signature_version = "1"
    timestamp = str(int(time.time()))

    string_to_sign = "\n".join([http_method, http_uri, ACCESS_KEY, data_type, signature_version, timestamp])
    sign = base64.b64encode(hmac.new(ACCESS_SECRET.encode('utf-8'),
                                     string_to_sign.encode('utf-8'),
                                     digestmod=hashlib.sha1).digest()).decode('utf-8')

    files = {'sample': open(file_path, 'rb')}
    payload = {
        'access_key': ACCESS_KEY,
        'data_type': data_type,
        'signature_version': signature_version,
        'signature': sign,
        'timestamp': timestamp
    }

    response = requests.post(f'https://{HOST}{http_uri}', files=files, data=payload)
    data = response.json()

    if 'metadata' in data and 'music' in data['metadata']:
        music = data['metadata']['music'][0]
        song_name = music.get('title', 'UNKNOWN')
        artist = music.get('artists', [{'name':'UNKNOWN'}])[0]['name']
        return song_name, artist, True
    else:
        return 'UNKNOWN', 'UNKNOWN', False

# === 1. SEGMENTS LOOP ===
segment_files = sorted(os.listdir(SEGMENTS_DIR))
results = []

for file in segment_files:
    file_path = os.path.join(SEGMENTS_DIR, file)
    print(f"Recognizing: {file}")
    song, artist, ok = recognize_file(file_path)
    results.append((file, song, artist, ok))

# === 2. TXT YAZ ===
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    for r in results:
        index = int(r[0].split('_')[1].split('.')[0])
        start_sec = 9 + (index-1)*13
        end_sec = start_sec + 11
        check = '✓' if r[3] else '✗'
        f.write(f"{start_sec:05.2f} - {end_sec:05.2f} | {r[1]} - {r[2]} {check}\n")

print(f"Recognition completed. TXT saved: {OUTPUT_TXT}")

Recognizing: segment_1.wav
Recognizing: segment_10.wav
Recognizing: segment_100.wav
Recognizing: segment_11.wav
Recognizing: segment_12.wav
Recognizing: segment_13.wav
Recognizing: segment_14.wav
Recognizing: segment_15.wav
Recognizing: segment_16.wav
Recognizing: segment_17.wav
Recognizing: segment_18.wav
Recognizing: segment_19.wav
Recognizing: segment_2.wav
Recognizing: segment_20.wav
Recognizing: segment_21.wav
Recognizing: segment_22.wav
Recognizing: segment_23.wav
Recognizing: segment_24.wav
Recognizing: segment_25.wav
Recognizing: segment_26.wav
Recognizing: segment_27.wav
Recognizing: segment_28.wav
Recognizing: segment_29.wav
Recognizing: segment_3.wav
Recognizing: segment_30.wav
Recognizing: segment_31.wav
Recognizing: segment_32.wav
Recognizing: segment_33.wav
Recognizing: segment_34.wav
Recognizing: segment_35.wav
Recognizing: segment_36.wav
Recognizing: segment_37.wav
Recognizing: segment_38.wav
Recognizing: segment_39.wav
Recognizing: segment_4.wav
Recognizing: segment_40